In [1]:
import os
import shutil
from sklearn.model_selection import train_test_split
from collections import defaultdict

base_dir = "/kaggle/input/datasets/joydippaul/mpox-skin-lesion-dataset-version-20-msld-v20/Original Images/Original Images/FOLDS/fold1"

# Các tập gốc
original_sets = ["Train", "Test", "Valid"]

# Thư mục output mới
output_dir = "/kaggle/working/re_split_dataset"

# Tỉ lệ chia mới
train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

# Seed để reproducible
random_seed = 42

# =========================
# TẠO THƯ MỤC OUTPUT
# =========================
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(output_dir, split), exist_ok=True)

# =========================
# LẤY DANH SÁCH CLASS
# =========================
classes = sorted(os.listdir(os.path.join(base_dir, "Train")))

print("Classes:", classes)

# =========================
# GỘP DỮ LIỆU TỪ 3 TẬP
# =========================
stats = defaultdict(dict)

for cls in classes:

    all_images = []

    # Gộp ảnh từ Train/Test/Valid
    for split in original_sets:

        class_dir = os.path.join(base_dir, split, cls)

        if os.path.exists(class_dir):

            images = [
                os.path.join(class_dir, img)
                for img in os.listdir(class_dir)
                if img.lower().endswith(('.png', '.jpg', '.jpeg'))
            ]

            all_images.extend(images)

    print(f"\n{cls}: Tổng số ảnh = {len(all_images)}")

    # =========================
    # CHIA TRAIN / VAL / TEST
    # =========================

    train_imgs, temp_imgs = train_test_split(
        all_images,
        test_size=(1 - train_ratio),
        random_state=random_seed,
        shuffle=True
    )

    val_imgs, test_imgs = train_test_split(
        temp_imgs,
        test_size=test_ratio / (test_ratio + val_ratio),
        random_state=random_seed,
        shuffle=True
    )

    # =========================
    # COPY FILE
    # =========================

    split_data = {
        "train": train_imgs,
        "val": val_imgs,
        "test": test_imgs
    }

    for split_name, img_list in split_data.items():

        target_class_dir = os.path.join(output_dir, split_name, cls)
        os.makedirs(target_class_dir, exist_ok=True)

        for img_path in img_list:

            filename = os.path.basename(img_path)
            dst_path = os.path.join(target_class_dir, filename)

            shutil.copy2(img_path, dst_path)

        stats[cls][split_name] = len(img_list)

# =========================
# THỐNG KÊ
# =========================
print("\n" + "="*50)
print("THỐNG KÊ SỐ LƯỢNG ẢNH")
print("="*50)

total_stats = {
    "train": 0,
    "val": 0,
    "test": 0
}

for cls in classes:

    print(f"\nClass: {cls}")

    for split in ["train", "val", "test"]:

        count = stats[cls][split]

        print(f"  {split.upper():5s}: {count}")

        total_stats[split] += count

print("\n" + "="*50)
print("TỔNG TOÀN DATASET")
print("="*50)

for split in ["train", "val", "test"]:
    print(f"{split.upper():5s}: {total_stats[split]}")

print("\nDataset mới được lưu tại:")
print(output_dir)

Classes: ['Chickenpox', 'Cowpox', 'HFMD', 'Healthy', 'Measles', 'Monkeypox']

Chickenpox: Tổng số ảnh = 75

Cowpox: Tổng số ảnh = 66

HFMD: Tổng số ảnh = 161

Healthy: Tổng số ảnh = 114

Measles: Tổng số ảnh = 55

Monkeypox: Tổng số ảnh = 284

THỐNG KÊ SỐ LƯỢNG ẢNH

Class: Chickenpox
  TRAIN: 60
  VAL  : 7
  TEST : 8

Class: Cowpox
  TRAIN: 52
  VAL  : 7
  TEST : 7

Class: HFMD
  TRAIN: 128
  VAL  : 16
  TEST : 17

Class: Healthy
  TRAIN: 91
  VAL  : 11
  TEST : 12

Class: Measles
  TRAIN: 44
  VAL  : 5
  TEST : 6

Class: Monkeypox
  TRAIN: 227
  VAL  : 28
  TEST : 29

TỔNG TOÀN DATASET
TRAIN: 602
VAL  : 74
TEST : 79

Dataset mới được lưu tại:
/kaggle/working/re_split_dataset


In [2]:
import shutil

# =========================
# TÊN FILE ZIP
# =========================
zip_output_path = "/kaggle/working/re_split_dataset"

# =========================
# ZIP DATASET
# =========================
shutil.make_archive(
    zip_output_path,
    'zip',
    output_dir
)

print("Đã tạo file zip:")
print(zip_output_path + ".zip")

Đã tạo file zip:
/kaggle/working/re_split_dataset.zip
